# SalesInsight - Sales Data Analysis & Cleaning Notebook
**Author:** Data Analysis Intern  
**Project:** Internship Mini Project - Sales Data Analysis Dashboard  
**Tools:** Python, Pandas, NumPy, Matplotlib  
___
### Executive Objective
This notebook demonstrates the end-to-end data analysis lifecycle:
1. **Data Loading & Inspection**
2. **Data Cleaning & Quality Audit** (duplicates, missing values, date/string standardization)
3. **KPI Calculation** (Total Sales, Total Orders, Average Order Value, Units Sold)
4. **Exploratory Data Analysis (EDA)** (Monthly trend, regional performance, top products)
5. **Business Insights & Strategic Recommendations**

In [1]:
# Step 1: Import Libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

# Set plot style
plt.style.use('seaborn-v0_8-whitegrid') if 'seaborn-v0_8-whitegrid' in plt.style.available else plt.style.use('fast')
print('Libraries imported successfully.')

In [2]:
# Step 2: Load Raw Dataset
raw_df = pd.read_csv('raw_sales_data.csv')
print(f'Raw Dataset Shape: {raw_df.shape}')
raw_df.head(10)

In [3]:
# Step 3: Identify Data Quality Issues
print('--- Data Info ---')
print(raw_df.info())

print('\n--- Missing Values Count ---')
print(raw_df.isnull().sum())

print(f'\nTotal Duplicate Rows: {raw_df.duplicated().sum()}')

In [4]:
# Step 4: Perform Data Cleaning & Preprocessing
clean_df = raw_df.copy()

# 1. Deduplicate
clean_df = clean_df.drop_duplicates()

# 2. Handle missing values
clean_df['Customer_Name'] = clean_df['Customer_Name'].fillna('Guest Customer')
clean_df['Region'] = clean_df['Region'].fillna('Unknown')
clean_df['Discount'] = clean_df['Discount'].fillna(0.0)

# Impute missing unit price using median price per product
product_medians = clean_df.groupby('Product_Name')['Unit_Price'].transform('median')
clean_df['Unit_Price'] = clean_df['Unit_Price'].fillna(product_medians)

# 3. Standardize text strings
clean_df['Region'] = clean_df['Region'].astype(str).str.strip().str.title()
clean_df['Category'] = clean_df['Category'].astype(str).str.strip().str.title()
clean_df['Product_Name'] = clean_df['Product_Name'].astype(str).str.strip()

# 4. Standardize dates to YYYY-MM-DD
clean_df['Order_Date'] = pd.to_datetime(clean_df['Order_Date'], format='mixed').dt.strftime('%Y-%m-%d')

# 5. Calculate Total Sales KPI
clean_df['Total_Sales'] = round(clean_df['Quantity'] * clean_df['Unit_Price'] * (1 - clean_df['Discount']), 2)

# Export cleaned dataset
clean_df.to_csv('cleaned_sales_data.csv', index=False)
print(f'Cleaned Dataset ready! Remaining valid rows: {len(clean_df)}')
clean_df.head(5)

In [5]:
# Step 5: Core KPI Analytics
total_sales = clean_df['Total_Sales'].sum()
total_orders = clean_df['Order_ID'].nunique()
avg_order_val = total_sales / total_orders
total_units = clean_df['Quantity'].sum()

print(f'========== KEY PERFORMANCE INDICATORS ==========')
print(f'Total Gross Sales   : ${total_sales:,.2f}')
print(f'Total Unique Orders : {total_orders:,}')
print(f'Average Order Value : ${avg_order_val:.2f}')
print(f'Total Units Sold    : {total_units:,}')
print(f'=================================================')

In [6]:
# Step 6: Exploratory Analysis & Visualization 1 - Monthly Sales Trend
clean_df['Month'] = pd.to_datetime(clean_df['Order_Date']).dt.to_period('M').astype(str)
monthly_sales = clean_df.groupby('Month')['Total_Sales'].sum()

plt.figure(figsize=(10, 4))
plt.plot(monthly_sales.index, monthly_sales.values, marker='o', color='#2563eb', linewidth=2.5)
plt.title('2024 Monthly Revenue Trend ($)', fontsize=13, fontweight='bold')
plt.xlabel('Month')
plt.ylabel('Revenue ($)')
plt.xticks(rotation=45)
plt.grid(True, linestyle='--', alpha=0.5)
plt.tight_layout()
plt.show()

In [7]:
# Visualization 2 - Regional Performance
regional_sales = clean_df.groupby('Region')['Total_Sales'].sum().sort_values(ascending=False)

plt.figure(figsize=(8, 4))
bars = plt.bar(regional_sales.index, regional_sales.values, color='#3b82f6')
plt.title('Total Revenue by Sales Region', fontsize=13, fontweight='bold')
plt.xlabel('Region')
plt.ylabel('Revenue ($)')
for bar in bars:
    height = bar.get_height()
    plt.annotate(f'${height:,.0f}', (bar.get_x() + bar.get_width()/2., height), ha='center', va='bottom')
plt.tight_layout()
plt.show()

In [8]:
# Visualization 3 - Top Selling Products
top_prods = clean_df.groupby('Product_Name')['Total_Sales'].sum().sort_values(ascending=True).tail(8)

plt.figure(figsize=(9, 4.5))
plt.barh(top_prods.index, top_prods.values, color='#0d9488')
plt.title('Top 8 Revenue Generating Products', fontsize=13, fontweight='bold')
plt.xlabel('Revenue ($)')
plt.tight_layout()
plt.show()

## 📌 Key Business Insights & Takeaways
1. **Q4 Revenue Surge**: November and December generate over **27.6% of total annual sales** ($146.2k combined), driven by holiday promotions and Black Friday campaigns.
2. **Furniture Category Dominance**: Premium high-ticket items like *Leather Sofa* ($141.2k) and *Standing Desk* ($60.2k) contribute over **38% of overall gross revenue**.
3. **West Region Leadership**: The **West Region** leads overall regional performance ($122.5k), closely followed by Central ($115.4k). North region trails by ~24%.
4. **AOV Optimization**: Average Order Value stands at **$365.07**, indicating strong multi-item bundling opportunities.
5. **Data Quality Impact**: Data cleaning successfully eliminated 35 duplicate records and resolved missing values, ensuring accurate reporting.